# AQI Hackathon — Notebook 1: Data Exploration

**Objective:** Understand the dataset before building any model.  
**Deliverable:** This notebook with every cell executed and your written observations in each markdown block.

Every section ends with a **Your Observation** block — fill it in with what you found.

---
## 0. Setup

In [1]:
# RUN
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
print('Ready.')

Ready.


---
## 1. Load Data
Upload `train.csv` and `test.csv` via the Files panel on the left, then run the cell below.

In [3]:
# RUN
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])

print(f'Train : {train.shape[0]:,} rows, {train.shape[1]} columns')
print(f'Test  : {test.shape[0]:,} rows, {test.shape[1]} columns')
print(f'Train period : {train["Date"].min().date()} → {train["Date"].max().date()}')
print(f'Test  period : {test["Date"].min().date()} → {test["Date"].max().date()}')
print(f'\nCities ({train["City"].nunique()}): {sorted(train["City"].unique())}')
train.head()

Train : 17,402 rows, 11 columns
Test  : 2,993 rows, 10 columns
Train period : 2015-01-29 → 2019-12-31
Test  period : 2020-01-01 → 2020-07-01

Cities (19): ['Ahmedabad', 'Amaravati', 'Amritsar', 'Bengaluru', 'Brajrajnagar', 'Chennai', 'Delhi', 'Gurugram', 'Guwahati', 'Hyderabad', 'Jaipur', 'Jorapokhar', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna', 'Talcher', 'Thiruvananthapuram', 'Visakhapatnam']


,City,StationId,StationName,Date,year,month,day_of_week,season,PM2.5,NO2,AQI_Bucket
0,Amaravati,AP001,"Secretariat, Amaravati - APPCB",2017-11-25,2017,11,5,Post-Monsoon,81.40,20.50,Moderate
1,Amaravati,AP001,"Secretariat, Amaravati - APPCB",2017-11-26,2017,11,6,Post-Monsoon,78.32,26.00,Moderate
2,Amaravati,AP001,"Secretariat, Amaravati - APPCB",2017-11-27,2017,11,0,Post-Monsoon,88.76,30.85,Moderate
3,Amaravati,AP001,"Secretariat, Amaravati - APPCB",2017-11-28,2017,11,1,Post-Monsoon,64.18,28.07,Moderate
4,Amaravati,AP001,"Secretariat, Amaravati - APPCB",2017-11-29,2017,11,2,Post-Monsoon,72.47,23.20,Moderate


In [4]:
# RUN — basic statistics
train[['PM2.5', 'NO2']].describe().round(2)

,PM2.5,NO2
count,17402.00,17402.00
mean,76.21,33.35
std,71.09,30.95
min,0.15,0.01
25%,32.79,12.72
50%,54.74,23.46
75%,91.41,43.29
max,995.00,448.05


---
## 2. Target Variable Distribution
Before doing anything else, understand what you are predicting.

> ⚠️ **Note on class imbalance:** Real-world datasets rarely have equal numbers of samples per class.
> The AQI dataset is no exception. Take note of which classes are frequent and which are rare

In [ ]:
# RUN
order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
colors = ['#4CAF50','#8BC34A','#FFC107','#FF9800','#F44336','#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = train['AQI_Bucket'].value_counts().reindex(order)

axes[0].bar(order, counts.values, color=colors, edgecolor='white')
axes[0].set_title('AQI Bucket Count', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Days'); axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontsize=9)

axes[1].pie(counts.values, labels=order, colors=colors,
            autopct='%1.1f%%', startangle=90, pctdistance=0.8)
axes[1].set_title('AQI Bucket Share', fontsize=12, fontweight='bold')

plt.tight_layout(); plt.show()
print(counts.to_string())

**Your Observation:**



---
## 3. City-Level Profiles
India has 19 cities in this dataset spread across very different geographies and climates.
Understanding city-level differences is essential.

In [ ]:
# RUN — median PM2.5 and NO2 per city
city_stats = (train.groupby('City')[['PM2.5','NO2']]
              .median().sort_values('PM2.5', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, clr in zip(axes, ['PM2.5','NO2'], ['tomato','steelblue']):
    city_stats[col].plot(kind='bar', ax=ax, color=clr, edgecolor='white')
    ax.set_title(f'Median {col} by City', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{col} (µg/m³)'); ax.tick_params(axis='x', rotation=45)
    ax.axhline(city_stats[col].mean(), color='black',
               linestyle='--', linewidth=1, label='Overall median')
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Your Observation:**



---
## 4. Temporal Patterns
Pollution does not stay constant throughout the year.
Explore how it changes over time.

In [ ]:
# RUN — overall monthly trend
monthly = train.groupby('month')['PM2.5'].median()
labels  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(range(1,13), monthly.values, alpha=0.15, color='tomato')
ax.plot(range(1,13), monthly.values, marker='o', color='tomato')
ax.set_xticks(range(1,13)); ax.set_xticklabels(labels)
ax.set_title('Median PM2.5 by Month (all cities)', fontsize=12, fontweight='bold')
ax.set_ylabel('PM2.5 (µg/m³)'); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

In [ ]:
# YOUR CODE



**Your Observation:**



---
## 5. PM2.5 and NO2 — Relationship to AQI
You are given both PM2.5 and NO2 as your sensor inputs.
Understand how they individually and jointly relate to the AQI category.

In [ ]:
# YOUR CODE


**Your Observation:**

---
## 6. Your Own Exploration
The provided plots cover the basics. Real data science rewards curiosity.

Add **at least two original explorations** driven by questions you formed while working through the sections above.
They can be anything — a new variable, a different cut of the data, an anomaly you noticed.
Do not repeat analyses already covered above.

The more you explore, better you models can be

In [ ]:
# YOUR CODE — Original exploration 1

**Your Observation:**

*What did you find?*

In [ ]:
# YOUR CODE — Original exploration 2

**Your Observation:**

*What did you find?*